<a href="https://colab.research.google.com/github/Pigwen/hands-on-sft/blob/main/Chapter_3_Low_Rank_Adaptation_(LoRA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
from copy import deepcopy
from numpy.linalg import matrix_rank
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Low-Rank Adaptation in a Nutshell

In [1]:
from torch import nn

base_layer = nn.Linear(1024, 1024, bias=False)
base_layer.weight.shape, base_layer.weight.numel()

(torch.Size([1024, 1024]), 1048576)

In [2]:
import torch

torch.manual_seed(11)
rank = 8
layer_A = nn.Linear(base_layer.in_features, rank, bias=False)
layer_B = nn.Linear(rank, base_layer.out_features, bias=False)
layer_A, layer_B

(Linear(in_features=1024, out_features=8, bias=False),
 Linear(in_features=8, out_features=1024, bias=False))

In [3]:
layer_A.weight.numel(), layer_B.weight.numel()

(8192, 8192)

In [5]:
composite = layer_B.weight @ layer_A.weight
composite.shape, composite.numel()

(torch.Size([1024, 1024]), 1048576)

In [6]:
from numpy.linalg import matrix_rank

matrix_rank(composite.detach().numpy())

np.int64(8)

In [9]:
torch.manual_seed(19)
batch = torch.randn(1, 1024)
batch @ (base_layer.weight + layer_B.weight @ layer_A.weight).T

tensor([[-0.0349,  0.6375,  0.0586,  ...,  0.1895, -0.2523,  0.8080]],
       grad_fn=<MmBackward0>)